# `farmland-mpc` — Colab demo

**Model-based AI planning for county-scale farmland consolidation in fragmented mountain landscapes.**

This notebook installs the `farmland_mpc` package and runs its end-to-end Phase A pipeline (DEM → slope → per-parcel zonal mean) on a 4-polygon synthetic fixture. No proprietary GIS dependencies; runs entirely in Colab.

Companion to: *Model-based AI planning enables county-scale farmland consolidation in fragmented mountain landscapes* (Paper 9 v7, in submission).

Repository: https://github.com/zhouning/arcgis-farmland-mpc

---

## 1. Install dependencies

Colab pre-installs `numpy`, `scipy`, `matplotlib`, `scikit-learn`, `networkx`, `torch`. We add the geospatial stack and pull `farmland-mpc` from GitHub.

In [ ]:
!pip install --quiet geopandas rasterio pyogrio shapely fiona libpysal typer tqdm onnx onnxruntime gymnasium

In [ ]:
# Install farmland-mpc from the public GitHub repo.
# If the repo is still private, use a personal access token:
#   !pip install --quiet git+https://<token>@github.com/zhouning/arcgis-farmland-mpc.git
!pip install --quiet git+https://github.com/zhouning/arcgis-farmland-mpc.git

In [ ]:
import farmland_mpc
import torch, rasterio, geopandas as gpd

print(f'farmland_mpc: {farmland_mpc.__version__}')
print(f'torch:        {torch.__version__}')
print(f'rasterio:     {rasterio.__version__}')
print(f'geopandas:    {gpd.__version__}')

## 2. Synthesise a toy region

We build a 100×100 tilted-plane DEM with a Gaussian bump in the top-right quadrant, plus a 4-polygon DLTB (one polygon per quadrant). Coordinate system: UTM Zone 48N (PROJ-string, bypasses the EPSG database).

In [ ]:
import tempfile, json
from pathlib import Path
import numpy as np
import geopandas as gpd
import rasterio
from rasterio.transform import from_origin
from shapely.geometry import box

CRS_WKT = (
    'PROJCRS["WGS 84 / UTM zone 48N",'
    'BASEGEOGCRS["WGS 84",DATUM["World Geodetic System 1984",'
    'ELLIPSOID["WGS 84",6378137,298.257223563,LENGTHUNIT["metre",1]]],'
    'PRIMEM["Greenwich",0,ANGLEUNIT["degree",0.0174532925199433]]],'
    'CONVERSION["UTM zone 48N",METHOD["Transverse Mercator",ID["EPSG",9807]],'
    'PARAMETER["Latitude of natural origin",0,ANGLEUNIT["degree",0.0174532925199433]],'
    'PARAMETER["Longitude of natural origin",105,ANGLEUNIT["degree",0.0174532925199433]],'
    'PARAMETER["Scale factor at natural origin",0.9996,SCALEUNIT["unity",1]],'
    'PARAMETER["False easting",500000,LENGTHUNIT["metre",1]],'
    'PARAMETER["False northing",0,LENGTHUNIT["metre",1]]],'
    'CS[Cartesian,2],'
    'AXIS["easting",east,ORDER[1],LENGTHUNIT["metre",1]],'
    'AXIS["northing",north,ORDER[2],LENGTHUNIT["metre",1]]]'
)

work = Path(tempfile.mkdtemp(prefix='farmland_mpc_colab_'))
dem_path  = work / 'dem.tif'
dltb_path = work / 'dltb.shp'
prepared  = work / 'prepared'

# Synthetic DEM
W = H = 100
cell = 30.0
xs, ys = np.arange(W)*cell, np.arange(H)*cell
xx, yy = np.meshgrid(xs, ys)
z = 100 + 0.05*xx + 0.02*yy
bump = 50.0 * np.exp(-((xx - 0.75*W*cell)**2 + (yy - 0.25*H*cell)**2) / (2*(10*cell)**2))
z = z + bump
ox, oy = 500000.0, 4400000.0
tf = from_origin(ox, oy, cell, cell)
with rasterio.open(dem_path, 'w', driver='GTiff', dtype='float32', nodata=-9999.0,
                   width=W, height=H, count=1, crs=CRS_WKT, transform=tf) as dst:
    dst.write(z.astype('float32'), 1)

# Synthetic DLTB (4 quadrant polygons)
hx = ox + (W//2)*cell
ht = oy - (H//2)*cell
by_y = oy - H*cell
rx = ox + W*cell
polys = [
    box(ox, ht, hx, oy),    # top-left  P001
    box(hx, ht, rx, oy),    # top-right P002 (bump)
    box(ox, by_y, hx, ht),  # bot-left  P003
    box(hx, by_y, rx, ht),  # bot-right P004
]
rows = [
    {'BSM':'P001','DLBM':'011','QSDWDM':'500227001','geometry':polys[0]},
    {'BSM':'P002','DLBM':'011','QSDWDM':'500227001','geometry':polys[1]},
    {'BSM':'P003','DLBM':'012','QSDWDM':'500227002','geometry':polys[2]},
    {'BSM':'P004','DLBM':'012','QSDWDM':'500227002','geometry':polys[3]},
]
gpd.GeoDataFrame(rows, crs=CRS_WKT).to_file(dltb_path, driver='ESRI Shapefile', encoding='utf-8')
print('Wrote:', dem_path.name, 'and', dltb_path.name, 'to', work)

## 3. Run Phase A: DEM → per-parcel slope

`farmland_mpc.prepare.run` reprojects the DEM to the target CRS, derives 3×3 Horn slope, then computes per-parcel zonal mean.

In [ ]:
from farmland_mpc.prepare import run

out_shp = run(
    dltb_path=dltb_path,
    dem_path=dem_path,
    prepared_dir=prepared,
    proj_crs=CRS_WKT,
)
print('Output shapefile:', out_shp)
summary = json.loads((prepared / 'prepare_data_summary.json').read_text(encoding='utf-8'))
print('Provenance:', json.dumps(summary, indent=2))

## 4. Inspect & visualise the result

Expect `P002` (top-right, with the Gaussian bump) to have the highest mean slope.

In [ ]:
out = gpd.read_file(out_shp)
print(out[['BSM','DLBM','slope_mean']])
assert out['slope_mean'].notna().all()
assert out.loc[out['BSM']=='P002','slope_mean'].iloc[0] > out.loc[out['BSM']=='P001','slope_mean'].iloc[0], \
    'expected P002 (bump quadrant) to have highest slope'
print('\nSmoke test PASS ✓')

In [ ]:
import matplotlib.pyplot as plt
from rasterio.plot import show

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

with rasterio.open(dem_path) as src:
    show(src, ax=axes[0], cmap='terrain')
axes[0].set_title('Synthetic DEM (m)')

slope_tif = prepared / 'dem_slope_analysis' / 'output' / '_slope_deg.tif'
with rasterio.open(slope_tif) as src:
    show(src, ax=axes[1], cmap='magma')
axes[1].set_title('Horn 3×3 slope (°)')

out.plot(column='slope_mean', cmap='magma', edgecolor='white', linewidth=2,
         legend=True, ax=axes[2])
for _, row in out.iterrows():
    c = row.geometry.centroid
    axes[2].annotate(f"{row['BSM']}\n{row['slope_mean']:.2f}°", (c.x, c.y),
                     ha='center', va='center', color='white', fontsize=9)
axes[2].set_title('Per-parcel mean slope')
axes[2].set_aspect('equal')

plt.tight_layout()
plt.show()

## 5. Next steps

This demo only ran Phase A (data preparation). The full pipeline has four stages:

1. **Prepare**: DEM + DLTB → per-parcel slope (this notebook).
2. **Sample**: random-policy transitions on the block-level MDP → contrastive pairs.
3. **Train**: 3-member transition-model ensemble with MSE + pairwise margin ranking loss.
4. **Plan**: MPC rollout against the learned world-model → optimised land-use plan.

On the published 2,600-block Bishan benchmark, the full pipeline achieves slope −1.54 % ± 0.04 % in **25 minutes of CPU compute**, compared with 8–12 GPU-hours for a comparable multi-agent RL baseline.

To run the full pipeline locally:

```bash
conda env create -f environment.yml
conda activate farmland-mpc
farmland-mpc prepare --dltb DLTB.shp --dem DEM.tif --prepared-dir ./out
farmland-mpc sample  --prepared-dir ./out
farmland-mpc train   --prepared-dir ./out
farmland-mpc plan    --prepared-dir ./out
```

An ArcGIS Pro toolbox wrapper (`LandUseOptimization_P9.pyt`) is also provided for users who prefer a GUI.

**Citation**:

> Zhou, N. *Model-based AI planning enables county-scale farmland consolidation in fragmented mountain landscapes.* In submission, 2026.